In [8]:
# Importing relevant libraries
import os, glob
import pandas as pd

In [9]:
# Setting path to the data
RAW="../data/raw"

In [10]:
# Creating 9 data frames, representing each dataset
dfs={
    os.path.basename(f).replace("olist_", "").replace("_dataset", "").replace(".csv", ""): pd.read_csv(f)
    for f in glob.glob(os.path.join(RAW, "*.csv"))
}
print(dfs.keys())

dict_keys(['sellers', 'product_category_name_translation', 'orders', 'order_items', 'customers', 'geolocation', 'order_payments', 'order_reviews', 'products'])


In [ ]:
def profile(name, d, key=None):
    print(f"==== {name} {d.shape}")
    print("null:\n", d.isna().sum()[d.isna().sum()>0]) # Finding the number of field with null values
    print("exact duplicate rows: ", d.duplicated().sum())
    if key:
        print(f"{key} unique?", d[key].is_unique)
    print()


profile("orders", dfs["orders"], "order_id")
profile("order_items", dfs["order_items"], "order_id")
profile("order_payments", dfs["order_payments"], "order_id")
profile("products", dfs["products"], "product_id")
profile("geolocation", dfs["geolocation"])
profile("order_reviews", dfs["order_reviews"],  "order_id")
profile("product_category_name_translation", dfs["product_category_name_translation"])
profile("sellers", dfs["sellers"], "seller_id")
profile("customers", dfs["customers"], "customer_id")

c=dfs["customers"]
print("Unique customers: ", c["customer_id"].nunique())

In [ ]:
# Creating a copy of raw data: 
o=dfs["orders"].copy()
# Cleaning up the dates
dates=["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date", "order_purchase_timestamp"]
# Looping through the columns in dates: 
for col in dates:
    o[col]=pd.to_datetime(o[col]) # Changing data type into dates
# Seeing new orders types now:
print(o.dtypes)
print("Total null: ", o[dates].isna().sum())

In [46]:
# Filtering data
print(o["order_status"].value_counts())
n0=len(o) # Keep track of starting point
# check delivered orders
o=o[o["order_status"]=="delivered"]
# Cleaning out any values without date
o=o.dropna(subset=["order_delivered_customer_date"])
# Seeing result
print(f"kept {len(o)} of {n0} orders ({len(o)/n0:.1%})")
# Checking results
print("Unique order status", o["order_status"].unique())
print("Delivered products with no delivered date", o["order_delivered_customer_date"].isna().sum())

order_status
delivered    96470
Name: count, dtype: int64
kept 96470 of 96470 orders (100.0%)
Unique order status <StringArray>
['delivered']
Length: 1, dtype: str
Delivered products with no delivered date 0


In [ ]:
""